<a href="https://colab.research.google.com/github/najouaelfethi/affiliations_um6p/blob/main/affiliation_um6p.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:

import json
import re
import time
from pathlib import Path
import pandas as pd
import requests

TEST_LIMIT = None  # set to an integer to test on the first N rows

FILE_PATH = Path("data") / "CCSE Publication List_oct2026.xlsx"

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_EXCEL = OUTPUT_DIR / "UM6P_affiliated_studies.xlsx"
CHECKPOINT_FILE = OUTPUT_DIR / "openalex_checkpoint.json"
FULL_JSON_FILE = OUTPUT_DIR / "openalex_full_responses.json"

OPENALEX_URL = (
    "https://api.openalex.org/works/https://doi.org/"
)

HEADERS = {
    "User-Agent": "UM6P-Affiliation-Research/1.0"
}

MAX_RETRIES = 4
REQUEST_DELAY = 0.15
CHECKPOINT_EVERY = 10

# Known names for UM6P.
# Add another verified variant here if needed.
UM6P_VARIANTS = [
    "UM6P",
    "U.M.6.P.",
    "Mohammed VI Polytechnic University",
    "Mohammed VI Polytechnic",
    "Mohamed VI Polytechnic University",
    "University Mohammed VI Polytechnic",
    "University of Mohammed VI Polytechnic",
    "University of Mohamed VI Polytechnic",
    "Université Mohammed VI Polytechnique",
    "Universite Mohammed VI Polytechnique",
    "Université Mohammed VI Polytechnic",
    "Universite Mohammed VI Polytechnic",
    "Mohammed VI Polytechnique University",
    "Mohamed VI Polytechnique University",
    "Mohammed VI Polytechnic School",
    "Mohammed VI Polytechnic Institute",
    "Mohammed 6 Polytechnic",
    "Mohamed 6 Polytechnic",
    "Mohammed 6 Polytechnique",
    "Mohamed 6 Polytechnique",
    "Mohammed VI Polytech",
    "Mohamed VI Polytech",
]


ENTITY_ALIASES = {
    "ACER CoE": [
        "Applied Chemistry and Engineering Research Centre of Excellence",
        "ACER CoE",
    ],
    "ARC_AIR": [
        "African Research Center on Air Quality and Climate",
        "ARC_Air",
    ],
    "CBS - GPE": [
        "Chemical & Biochemical Sciences",
        "Chemical and Biochemical Sciences",
        "Green Process Engineering",
        "CBS",
        "GPE",
    ],
    "HTMR": [
        "High Throughput Multidisciplinary Research Laboratory",
        "HTMR",
    ],
    "IST&I": [
        "Institute of Science, Technology & Innovation",
        "Institute of Science, Technology and Innovation",
        "IST&I",
    ],
    "LIMSET": [
        "Laboratory of Inorganic Materials for Sustainable Energy Technologies",
        "LIMSET",
    ],
    "MSN": [
        "Department of Materials Science, Energy and Nano-engineering",
        "Materials Science, Energy and Nano-engineering",
        "Materials Science and Nano-Engineering Department",
        "Materials Science and Nano-Engineering",
        "Materials Science & Nano-Engineering",
        "Materials Science and Nanoengineering",
        "Materials Science, Energy and Nanoengineering",
        "MSN",
    ],
    "SUSMAT-RC": [
        "Sustainable Materials Research Center",
        "SusMat-RC",
        "SUSMAT-RC",
    ],
}


# 2. DOI NORMALIZATION

def normalize_doi(value):
    """Normalize a DOI or DOI URL from Excel."""

    if pd.isna(value):
        return None

    doi = str(value).strip()

    doi = re.sub(
        r"^(https?://)?(dx\.)?doi\.org/",
        "",
        doi,
        flags=re.IGNORECASE,
    )

    doi = re.sub(r"^doi:\s*", "", doi, flags=re.IGNORECASE)
    doi = doi.strip()

    if not doi or doi.casefold() in {"nan", "none", "-"}:
        return None

    return doi.lower()



def matches_um6p(value):
    """Check whether text contains a known UM6P name."""

    if not value:
        return False

    normalized = str(value).casefold()

    return any(
        variant.casefold() in normalized
        for variant in UM6P_VARIANTS
    )


def normalize_entity_text(text):
    """Normalize punctuation, case, and whitespace."""
    text = str(text).casefold()
    return re.sub(r"[^a-z0-9]+", " ", text).strip()


def classify_entities(affiliation):
    """Identify all specific entities mentioned in an affiliation."""

    normalized = normalize_entity_text(affiliation)
    matched = []

    for entity, aliases in ENTITY_ALIASES.items():
        for alias in aliases:
            normalized_alias = normalize_entity_text(alias)

            if re.search(
                rf"\b{re.escape(normalized_alias)}\b",
                normalized
            ):
                matched.append(entity)
                break

    return matched


def extract_um6p_lines(affiliation_text):
    """Keep only segments that mention UM6P."""

    if not affiliation_text:
        return []

    text = re.sub(
        r"<br\s*/?>", " ", str(affiliation_text), flags=re.I
    )
    text = re.sub(r"\s+", " ", text).strip()

    segments = []
    for part in text.split("||"):
        segments.extend(split_outside_brackets(part, ";"))

    cleaned = []
    for segment in segments:
        segment = re.sub(
            r"\.?\s*Electronic address:.*$", "", segment, flags=re.I
        ).strip(" .,")
        if segment and matches_um6p(segment):
            cleaned.append(segment)

    return list(dict.fromkeys(cleaned))


def split_outside_brackets(text, separator):
    """Split on a separator, ignoring ones inside (), [] or {}."""

    parts, current, depth = [], [], 0

    for char in text:
        if char in "([{":
            depth += 1
        elif char in ")]}":
            depth = max(depth - 1, 0)

        if char == separator and depth == 0:
            parts.append("".join(current))
            current = []
        else:
            current.append(char)

    parts.append("".join(current))
    return parts


# 3. CHECKPOINT MANAGEMENT

def load_checkpoint():
    """Load previous API results to avoid repeating requests."""

    if not CHECKPOINT_FILE.exists():
        return {}

    with open(CHECKPOINT_FILE, "r", encoding="utf-8") as file:
        return json.load(file)


def save_json(path, data):
    """Save JSON through a temporary file."""

    temporary_path = path.with_suffix(path.suffix + ".tmp")

    with open(temporary_path, "w", encoding="utf-8") as file:
        json.dump(data, file, ensure_ascii=False, indent=2)

    temporary_path.replace(path)


# 4. OPENALEX REQUEST

def fetch_openalex(session, doi):
    """Fetch the complete OpenAlex record for a DOI."""

    url = OPENALEX_URL + doi

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = session.get(url, timeout=30)

            if response.status_code == 404:
                return {
                    "status": "not_found",
                    "doi": doi,
                    "http_status": 404,
                }

            if (
                response.status_code == 429
                or response.status_code >= 500
            ):
                if attempt < MAX_RETRIES:
                    retry_after = response.headers.get("Retry-After")

                    try:
                        delay = float(retry_after)
                    except (TypeError, ValueError):
                        delay = 2 ** attempt

                    time.sleep(delay)
                    continue

            response.raise_for_status()

            return {
                "status": "success",
                "doi": doi,
                "http_status": response.status_code,
                "data": response.json(),
            }

        except (requests.RequestException, ValueError) as error:
            if attempt == MAX_RETRIES:
                return {
                    "status": "error",
                    "doi": doi,
                    "error": str(error),
                }

            time.sleep(2 ** attempt)

    return {
        "status": "error",
        "doi": doi,
        "error": "Maximum retries exceeded",
    }



# 6. PREPARE EXCEL OUTPUT

def build_output_row(row, result):
    """Keep important study fields and add UM6P results."""

    authors, status = extract_um6p_authors(result)

    entity_names = []
    entity_matches = []

    for author in authors:
        author_entities = set()

        for affiliation in author["affiliations"]:
            entities = classify_entities(affiliation)
            author_entities.update(entities)

            for entity in entities:
                if entity not in entity_names:
                    entity_names.append(entity)

                entity_matches.append({
                    "author": author["author"],
                    "entity": entity,
                    "affiliation": affiliation,
            })

        if not author_entities and "UM6P" not in entity_names:
            entity_names.append("UM6P")


    author_names = list(dict.fromkeys(
        author["author"] for author in authors
    ))

    all_affiliations = list(dict.fromkeys(
        affiliation
        for author in authors
        for affiliation in author["affiliations"]
    ))

    author_affiliation_details = [
        {
            "author": author["author"],
            "affiliations": author["affiliations"],
        }
        for author in authors
    ]


    return {
        "Title": row.get("Title", ""),
        "Year": row.get("Year", ""),
        "DOI": row.get("DOI", ""),
        "Publication type": row.get("Publication type", ""),
        "Scopus Source title": row.get("Scopus Source title", ""),
        "UM6P_Found": "Yes" if authors else (
            "No match in metadata" if status == "NO_MATCH_IN_OPENALEX"
            else "Unknown"
        ),
        "UM6P_Author_Count": len(author_names),
        "UM6P_Authors": "; ".join(author_names),
        "UM6P_Affiliations": " || ".join(all_affiliations),
        "UM6P_Entities": "; ".join(entity_names),
        "UM6P_Affiliation_Entity_Matches": json.dumps(entity_matches, ensure_ascii=False),  # dropped from main sheet
        "UM6P_Author_Affiliation_Details": json.dumps(author_affiliation_details,ensure_ascii=False),
    }


def extract_um6p_authors(result):
    """Extract UM6P authors with only UM6P-related affiliations."""

    if result is None:
        return [], "NOT_PROCESSED"

    if result.get("status") == "not_found":
        return [], "DOI_NOT_FOUND_IN_OPENALEX"

    if result.get("status") != "success":
        return [], "API_ERROR"

    work = result.get("data", {})
    authorships = work.get("authorships") or []

    if not authorships:
        return [], "NO_AUTHORSHIP_DATA"

    matches = []

    for authorship in authorships:
        author = authorship.get("author") or {}
        author_name = (
            authorship.get("raw_author_name")
            or author.get("display_name")
            or "Unknown author"
        )

        institutions = authorship.get("institutions") or []
        institution_names = {
            institution.get("id"): institution.get("display_name", "")
            for institution in institutions
            if institution.get("id")
        }

        matched_affiliations = []

        # Inspect structured affiliations.
        for affiliation in authorship.get("affiliations") or []:
            text = affiliation.get("raw_affiliation_string") or ""

            ids = affiliation.get("institution_ids") or []
            linked_names = [
                institution_names.get(institution_id, "")
                for institution_id in ids
            ]

            if (
                matches_um6p(text)
                or any(matches_um6p(name) for name in linked_names)
            ):
                matched_affiliations.extend(extract_um6p_lines(text))

                # Retain the linked UM6P institution name if the
                # raw affiliation text doesn't contain its name.
                if not extract_um6p_lines(text):
                    matched_affiliations.extend(
                        name for name in linked_names
                        if matches_um6p(name)
                    )

        # Always inspect raw affiliation strings too.
        for text in authorship.get("raw_affiliation_strings") or []:
            matched_affiliations.extend(extract_um6p_lines(text))

        matched_affiliations = list(dict.fromkeys(
            text for text in matched_affiliations if text
        ))

        if matched_affiliations:
            key = author_name.casefold().strip()
            existing = next(
                (m for m in matches if m["author"].casefold().strip() == key),
                None,
            )

            if existing:
                existing["affiliations"] = list(dict.fromkeys(
                    existing["affiliations"] + matched_affiliations
                ))
                existing["orcid"] = existing["orcid"] or (
                    author.get("orcid") or authorship.get("raw_orcid") or ""
                )
                continue

            matches.append({
                "author": author_name,
                "affiliations": matched_affiliations,
                "author_position": authorship.get("author_position", ""),
                "orcid": author.get("orcid") or authorship.get("raw_orcid") or "",
            })

    if matches:
        return matches, "MATCHED"

    return [], "NO_MATCH_IN_OPENALEX"


def export_entity_sheets(output_df, output_path):
    """Export eight entity sheets and one general UM6P sheet."""

    merged = {entity: {} for entity in ENTITY_ALIASES}
    merged["UM6P"] = {}

    for _, row in output_df.iterrows():
        details = row.get("UM6P_Author_Affiliation_Details", "")

        if not isinstance(details, str) or not details.strip():
            continue

        try:
            authors = json.loads(details)
        except (json.JSONDecodeError, TypeError):
            continue

        for author in authors:
            author_name = author.get("author", "Unknown author")
            author_has_entity = any(
                classify_entities(a) for a in author.get("affiliations", [])
            )

            for affiliation in author.get("affiliations", []):
                entities = classify_entities(affiliation)

                record = {
                    "Author": author_name,
                    "Affiliation": affiliation,
                    "Title": row.get("Title", ""),
                    "Year": row.get("Year", ""),
                    "DOI": row.get("DOI", ""),
                }

                if entities:
                    targets = entities
                elif not author_has_entity:
                    # Author has no recognized entity in any affiliation.
                    targets = ["UM6P"]
                else:
                    targets = []

                for entity in targets:
                    key = author_name.casefold().strip()
                    existing = merged[entity].get(key)

                    if existing is None:
                        merged[entity][key] = {
                            "Author": author_name,
                            "Affiliations": [affiliation],
                            "Publications": [(
                                str(record["Title"]),
                                str(record["Year"]),
                                str(record["DOI"]),
                            )],
                        }
                        continue

                    if affiliation not in existing["Affiliations"]:
                        existing["Affiliations"].append(affiliation)

                    publication = (
                        str(record["Title"]),
                        str(record["Year"]),
                        str(record["DOI"]),
                    )
                    if publication not in existing["Publications"]:
                        existing["Publications"].append(publication)

    entity_rows = {
        entity: [
            {
                "Author": item["Author"],
                "Affiliations": " || ".join(item["Affiliations"]),
                "Publication_Count": len(item["Publications"]),
                "Titles": " || ".join(p[0] for p in item["Publications"]),
                "Years": " || ".join(p[1] for p in item["Publications"]),
                "DOIs": " || ".join(p[2] for p in item["Publications"]),
            }
            for item in by_key.values()
        ]
        for entity, by_key in merged.items()
    }

    with pd.ExcelWriter(
        output_path,
        engine="openpyxl",
        mode="a",
        if_sheet_exists="replace",
    ) as writer:

        for entity, records in entity_rows.items():
            entity_df = pd.DataFrame(records).drop_duplicates()
            entity_df.to_excel(
                writer,
                sheet_name=entity[:31],
                index=False,
            )

    print("Entity sheets exported successfully.")
    for entity, records in entity_rows.items():
        print(f"{entity}: {len(pd.DataFrame(records).drop_duplicates())} rows")


def main():

    if not FILE_PATH.exists():
        raise FileNotFoundError(
            f"Excel file not found: {FILE_PATH.resolve()}"
        )

    # The actual column headers are on Excel row 17.
    df = pd.read_excel(FILE_PATH, header=16)

    df.columns = df.columns.astype(str).str.strip()
    df = df.dropna(how="all").reset_index(drop=True)
    if TEST_LIMIT:
        df = df.head(TEST_LIMIT).copy()

    if "DOI" not in df.columns:
        raise ValueError(
            f"DOI column not found. Available columns: {df.columns.tolist()}"
        )

    # Keep the original DOI values and normalize only for lookups.
    df["_normalized_doi"] = df["DOI"].apply(normalize_doi)

    unique_dois = df["_normalized_doi"].dropna().unique().tolist()

    print(f"Studies loaded: {len(df)}")
    print(f"Unique DOIs: {len(unique_dois)}")

    # Resume saved results.
    results = load_checkpoint()

    session = requests.Session()
    session.headers.update(HEADERS)

    try:
        pending_dois = [
            doi for doi in unique_dois
            if doi not in results
            or results[doi].get("status") == "error"
        ]

        for index, doi in enumerate(pending_dois, start=1):
            print(
                f"[{index}/{len(pending_dois)}] "
                f"Retrieving DOI: {doi}"
            )

            results[doi] = fetch_openalex(session, doi)

            if index % CHECKPOINT_EVERY == 0:
                save_json(CHECKPOINT_FILE, results)

            time.sleep(REQUEST_DELAY)

    except KeyboardInterrupt:
        print("\nInterrupted. Saving progress...")

    finally:
        session.close()
        save_json(CHECKPOINT_FILE, results)
        save_json(FULL_JSON_FILE, results)

    # Rebuild the output from all source rows.
    # Studies without DOIs are retained and marked unknown.
    output_rows = []

    for _, row in df.iterrows():
        doi = row["_normalized_doi"]

        result = results.get(doi) if doi else None

        output_row = build_output_row(row, result)

        if not doi:
            output_row["UM6P_Found"] = "Unknown"

        output_rows.append(output_row)

    output_df = pd.DataFrame(output_rows)

    # One row per study; authors, affiliations and entities are aligned
    # by position (one " || " block per author, even if identical).
    author_rows = []

    for _, row in output_df.iterrows():
        try:
            authors = json.loads(row["UM6P_Author_Affiliation_Details"])
        except (json.JSONDecodeError, TypeError):
            authors = []

        affiliation_blocks = []
        entity_blocks = []

        for author in authors:
            affiliations = author.get("affiliations", [])
            entities = list(dict.fromkeys(
                entity
                for affiliation in affiliations
                for entity in classify_entities(affiliation)
            )) or ["UM6P"]

            affiliation_blocks.append(" ; ".join(affiliations))
            entity_blocks.append(", ".join(entities))

        author_rows.append({
            "Title": row["Title"],
            "Year": row["Year"],
            "DOI": row["DOI"],
            "Publication type": row["Publication type"],
            "Scopus Source title": row["Scopus Source title"],
            "UM6P_Found": row["UM6P_Found"],
            "UM6P_Author_Count": len(authors),
            "UM6P_Authors": "; ".join(a.get("author", "") for a in authors),
            "UM6P_Affiliations": " || ".join(affiliation_blocks),
            "UM6P_Entities": " || ".join(entity_blocks),
        })

    # Save as a new Excel file. The original remains unchanged.
    with pd.ExcelWriter(OUTPUT_EXCEL, engine="openpyxl") as writer:
        pd.DataFrame(author_rows).to_excel(
            writer,
            sheet_name="UM6P Affiliations",
            index=False,
        )
    export_entity_sheets(output_df, OUTPUT_EXCEL)

    print("\nFinished.")
    print(f"Output Excel: {OUTPUT_EXCEL.resolve()}")
    print(f"Full JSON: {FULL_JSON_FILE.resolve()}")
    print(f"Checkpoint: {CHECKPOINT_FILE.resolve()}")

    print(
        "\nStudies with UM6P matches:",
        (output_df["UM6P_Found"] == "Yes").sum(),
    )


if __name__ == "__main__":
    main()


Studies loaded: 1509
Unique DOIs: 1505
[1/1505] Retrieving DOI: 10.1039/d2cs00569g
[2/1505] Retrieving DOI: 10.3390/molecules25215207
[3/1505] Retrieving DOI: 10.1038/s41467-022-30020-4
[4/1505] Retrieving DOI: 10.1016/j.jconrel.2020.12.026
[5/1505] Retrieving DOI: 10.1002/aenm.201702403
[6/1505] Retrieving DOI: 10.1016/j.jclepro.2023.137561
[7/1505] Retrieving DOI: 10.1002/anie.202116560
[8/1505] Retrieving DOI: 10.1016/j.ijbiomac.2017.08.067
[9/1505] Retrieving DOI: 10.1021/acsenergylett.1c01678
[10/1505] Retrieving DOI: 10.1155/2021/5518999
[11/1505] Retrieving DOI: 10.1016/j.ijhydene.2023.10.218
[12/1505] Retrieving DOI: 10.1016/j.indcrop.2017.12.049
[13/1505] Retrieving DOI: 10.1038/s41563-023-01545-w
[14/1505] Retrieving DOI: 10.1016/j.ijbiomac.2018.12.030
[15/1505] Retrieving DOI: 10.1016/j.carbpol.2020.116506
[16/1505] Retrieving DOI: 10.3390/polym15132908
[17/1505] Retrieving DOI: 10.1007/s10570-020-03097-7
[18/1505] Retrieving DOI: 10.1016/j.carbon.2018.05.032
[19/1505] Retri